# LightFM User Split (80/20) with KGE Features
This notebook preprocesses data, exports all intermediate outputs to one output folder, builds a compact cold-start focused Knowledge Graph, trains KGE embeddings, reloads artifacts, then trains LightFM with the same baseline configuration.

## 1. Environment Setup

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")

from pathlib import Path
import json
import re

if os.name == "nt":
    conda_prefix = os.environ.get("CONDA_PREFIX", "")
    dll_candidates = [
        Path(conda_prefix) / "Library" / "bin",
        Path(conda_prefix) / "DLLs",
        Path(conda_prefix) / "Lib" / "site-packages" / "torch" / "lib",
    ]
    dll_paths = [str(p) for p in dll_candidates if p.exists()]

    if dll_paths:
        os.environ["PATH"] = ";".join(dll_paths + [os.environ.get("PATH", "")])
        for dll_path in dll_paths:
            try:
                os.add_dll_directory(dll_path)
            except (FileNotFoundError, OSError):
                pass

import numpy as np
import pandas as pd

np.random.seed(42)

### Optional one-time dependency installation
Run only if required packages are missing in your current environment.

In [ ]:
# %pip install lightfm pykeen "torch==2.4.1" "torchvision==0.19.1" "torchaudio==2.4.1"

In [ ]:
import os
from pathlib import Path

if os.name == "nt":
    conda_prefix = os.environ.get("CONDA_PREFIX", "")
    dll_candidates = [
        Path(conda_prefix) / "Library" / "bin",
        Path(conda_prefix) / "DLLs",
        Path(conda_prefix) / "Lib" / "site-packages" / "torch" / "lib",
    ]
    dll_paths = [str(p) for p in dll_candidates if p.exists()]

    if dll_paths:
        os.environ["PATH"] = ";".join(dll_paths + [os.environ.get("PATH", "")])
        for dll_path in dll_paths:
            try:
                os.add_dll_directory(dll_path)
            except (FileNotFoundError, OSError):
                pass

import torch

print(f"Torch version: {torch.__version__}")
print(f"Torch CUDA build: {torch.version.cuda}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"GPU: {gpu_name} | VRAM: {vram_gb:.1f} GB")
else:
    print("No CUDA GPU detected. KGE training will run on CPU.")

Torch version: 2.4.1+cu124
Torch CUDA build: 12.4
CUDA available: True
GPU: NVIDIA GeForce RTX 3050 Laptop GPU | VRAM: 4.0 GB


## 2. Paths and Output Registry

In [ ]:
DATA_DIR = Path("data")
MOVIES_PATH = DATA_DIR / "movies.csv"
RATINGS_PATH = DATA_DIR / "ratings.csv"
USERS_PATH = DATA_DIR / "user_profiles.csv"

OUTPUT_DIR = Path("output") / "kge_cold_start"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_FILES = {
    "ratings_clean": OUTPUT_DIR / "01clean_ratings.csv",
    "movies_clean": OUTPUT_DIR / "01clean_movies.csv",
    "users_clean": OUTPUT_DIR / "01clean_users.csv",
    "movie_content_features": OUTPUT_DIR / "02movie_content_features.csv",
    "user_behavior_features": OUTPUT_DIR / "02user_behavior_features.csv",
    "train_ratings": OUTPUT_DIR / "03split_train_ratings.csv",
    "test_ratings": OUTPUT_DIR / "03split_test_ratings.csv",
    "train_keys": OUTPUT_DIR / "03split_train_keys.npy",
    "test_keys": OUTPUT_DIR / "03split_test_keys.npy",
    "all_item_ids": OUTPUT_DIR / "03split_all_item_ids.npy",
    "kg_triples": OUTPUT_DIR / "04kg_triples.tsv",
    "kg_relation_stats": OUTPUT_DIR / "04kg_relation_stats.csv",
    "kge_entity_to_id": OUTPUT_DIR / "05kge_entity_to_id.csv",
    "kge_relation_to_id": OUTPUT_DIR / "05kge_relation_to_id.csv",
    "kge_entity_embeddings": OUTPUT_DIR / "05kge_entity_embeddings.npy",
    "kge_entity_labels": OUTPUT_DIR / "05kge_entity_labels.npy",
    "kge_item_embeddings": OUTPUT_DIR / "05kge_item_embeddings.csv",
    "kge_user_embeddings": OUTPUT_DIR / "05kge_user_embeddings.csv",
    "metrics_json": OUTPUT_DIR / "06lightfm_test_metrics.json",
}

for required_path in [MOVIES_PATH, RATINGS_PATH, USERS_PATH]:
    if not required_path.exists():
        raise FileNotFoundError(f"Missing required file: {required_path}")

print(f"Input dir: {DATA_DIR.resolve()}")
print(f"Output dir: {OUTPUT_DIR.resolve()}")

Input dir: C:\Users\minhg\Desktop\New folder\data
Output dir: C:\Users\minhg\Desktop\New folder\output\kge_cold_start


## 3. Load, Clean, and Export Base Tables

In [ ]:
movies_df = pd.read_csv(MOVIES_PATH)
ratings_df = pd.read_csv(RATINGS_PATH)
users_df = pd.read_csv(USERS_PATH)

ratings = ratings_df.rename(columns={"id": "user_id", "rate": "rating"}).copy()
users = users_df.rename(columns={"id": "user_id"}).copy()
movies = movies_df.copy()

ratings["user_id"] = pd.to_numeric(ratings["user_id"], errors="coerce")
ratings["movie_id"] = pd.to_numeric(ratings["movie_id"], errors="coerce")
ratings["rating"] = pd.to_numeric(ratings["rating"], errors="coerce")
ratings = ratings[["user_id", "movie_id", "rating"]].dropna().copy()
ratings["user_id"] = ratings["user_id"].astype(int)
ratings["movie_id"] = ratings["movie_id"].astype(int)

movies["id"] = pd.to_numeric(movies["id"], errors="coerce")
movies = movies.dropna(subset=["id"]).copy()
movies["id"] = movies["id"].astype(int)
movies = movies.drop_duplicates(subset=["id"]).copy()

users["user_id"] = pd.to_numeric(users["user_id"], errors="coerce")
users = users.dropna(subset=["user_id"]).copy()
users["user_id"] = users["user_id"].astype(int)
users = users.drop_duplicates(subset=["user_id"]).copy()

for col in ["genres", "topics", "country_code", "plot", "main_title", "year_published"]:
    if col not in movies.columns:
        movies[col] = ""

movies["genres"] = movies["genres"].fillna("")
movies["topics"] = movies["topics"].fillna("")
movies["country_code"] = movies["country_code"].fillna("")
movies["plot"] = movies["plot"].fillna("")
movies["main_title"] = movies["main_title"].fillna("")
movies["year_published"] = pd.to_numeric(movies["year_published"], errors="coerce")

base_behavior_cols = [
    "early_bird",
    "night_owl",
    "weekend_tweeter",
    "week_tweeter",
    "geo_enabled",
]
behavior_cols = [c for c in base_behavior_cols if c in users.columns]

if "number_of_tweets" in users.columns:
    tweets = pd.to_numeric(users["number_of_tweets"], errors="coerce").fillna(0.0)
    tweet_threshold = float(tweets.quantile(0.75))
    users["heavy_tweeter"] = (tweets >= tweet_threshold).astype(int)
else:
    users["heavy_tweeter"] = 0
behavior_cols.append("heavy_tweeter")

if "followers_count" not in users.columns:
    users["followers_count"] = 0
users["followers_count"] = pd.to_numeric(users["followers_count"], errors="coerce").fillna(0.0)
users["log_followers"] = np.log1p(users["followers_count"])

def split_tokens(value):
    if pd.isna(value) or value == "":
        return []
    return [t.strip() for t in str(value).split("|") if t.strip()]

def canonical_pipe_text(value):
    items = split_tokens(value)
    if not items:
        return ""
    dedup = list(dict.fromkeys(items))
    return "|".join(dedup)

def sanitize_node_token(token):
    token = str(token).strip().lower()
    token = re.sub(r"\s+", "_", token)
    token = re.sub(r"[^a-z0-9_]+", "", token)
    return token or "unknown"

movies_tmp = movies.copy()
year_min = movies_tmp["year_published"].min(skipna=True)
year_max = movies_tmp["year_published"].max(skipna=True)
year_span = (year_max - year_min) if pd.notna(year_min) and pd.notna(year_max) and year_max != year_min else 1.0
movies_tmp["year_norm"] = ((movies_tmp["year_published"] - year_min) / year_span).fillna(0.0).clip(0.0, 1.0)

movie_content_features = pd.DataFrame({
    "movie_id": movies_tmp["id"].astype(int),
    "genres": movies_tmp["genres"].map(canonical_pipe_text),
    "topics": movies_tmp["topics"].map(canonical_pipe_text),
    "country_code": movies_tmp["country_code"].astype(str).str.upper().str.strip(),
    "year_norm": movies_tmp["year_norm"].astype(float),
    "has_plot": movies_tmp["plot"].astype(str).str.strip().ne("").astype(int),
})

user_behavior_features = users[["user_id", "log_followers"] + behavior_cols].copy()
for col in behavior_cols:
    user_behavior_features[col] = pd.to_numeric(user_behavior_features[col], errors="coerce").fillna(0).astype(int)

ratings.to_csv(OUTPUT_FILES["ratings_clean"], index=False)
movies.to_csv(OUTPUT_FILES["movies_clean"], index=False)
users.to_csv(OUTPUT_FILES["users_clean"], index=False)
movie_content_features.to_csv(OUTPUT_FILES["movie_content_features"], index=False)
user_behavior_features.to_csv(OUTPUT_FILES["user_behavior_features"], index=False)

print(f"Movies: {len(movies):,} | Ratings: {len(ratings):,} | Users: {len(users):,}")
print(f"Exported cleaned tables to: {OUTPUT_DIR}")

Movies: 78,628 | Ratings: 1,172,038 | Users: 482
Exported cleaned tables to: output\kge_cold_start


## 4. Train-Test Split by User ID (80/20) and Export

In [ ]:
def split_users_train_test(ratings_df, test_size=0.20, seed=42):
    unique_users = ratings_df["user_id"].dropna().astype(int).unique().copy()
    if len(unique_users) < 2:
        raise ValueError("At least two users are required for split.")

    rng = np.random.default_rng(seed)
    rng.shuffle(unique_users)

    n_train_users = int(round(len(unique_users) * (1.0 - test_size)))
    n_train_users = min(max(1, n_train_users), len(unique_users) - 1)

    train_users = unique_users[:n_train_users]
    test_users = unique_users[n_train_users:]

    train_df = ratings_df[ratings_df["user_id"].isin(train_users)].copy()
    test_df = ratings_df[ratings_df["user_id"].isin(test_users)].copy()
    return train_df, test_df, train_users, test_users

train_df, test_df, train_users, test_users = split_users_train_test(ratings, test_size=0.20, seed=42)
all_item_ids = sorted(movies["id"].dropna().astype(int).unique().tolist())

train_df.to_csv(OUTPUT_FILES["train_ratings"], index=False)
test_df.to_csv(OUTPUT_FILES["test_ratings"], index=False)
np.save(OUTPUT_FILES["train_keys"], np.array(train_users, dtype=np.int64))
np.save(OUTPUT_FILES["test_keys"], np.array(test_users, dtype=np.int64))
np.save(OUTPUT_FILES["all_item_ids"], np.array(all_item_ids, dtype=np.int64))

print(f"Split users -> train: {len(train_users):,}, test: {len(test_users):,}")
print(f"Split rows  -> train: {len(train_df):,}, test: {len(test_df):,}")
print(f"Exported split outputs to: {OUTPUT_DIR}")

Split users -> train: 489, test: 122
Split rows  -> train: 938,844, test: 233,194
Exported split outputs to: output\kge_cold_start


## 5. Knowledge Graph Schema Design for Cold Start

### Recommended minimal schema (important for cold start)
- Entities: user_*, movie_*, genre_*, topic_*, country_*, behavior_*
- Relations: rated, has_genre, has_topic, from_country, has_behavior
- Excluded by design: actors/directors (very high cardinality and sparse impact)

### How to prioritize by target
- User cold start: prioritize user behaviors + movie content (genre/topic/country), keep rated edges from warm users for graph connectivity.
- Item cold start: prioritize movie content edges (has_genre/has_topic/from_country) because new items have no ratings yet; behaviors remain useful as contextual bridge.

The next cells implement this compact schema and train TransE embeddings from these triples.

In [ ]:
COLD_START_FOCUS = "user"  # choose: "user" or "item"
POSITIVE_RATING_THRESHOLD = 7.0
MAX_TOPICS_PER_MOVIE = 5

if COLD_START_FOCUS not in {"user", "item"}:
    raise ValueError("COLD_START_FOCUS must be either 'user' or 'item'.")

## 6. Build Compact KG Triples and Export
This version adds fallback behavior edges so every test user has at least one KG edge for fair user cold-start comparison.

In [ ]:
train_ratings = pd.read_csv(OUTPUT_FILES["train_ratings"]).copy()
movies_clean = pd.read_csv(OUTPUT_FILES["movies_clean"]).copy()
user_behavior = pd.read_csv(OUTPUT_FILES["user_behavior_features"]).copy()
movie_content = pd.read_csv(OUTPUT_FILES["movie_content_features"]).copy()

# Use exported split keys so we can guarantee KG coverage for every test user.
train_users_split = set(np.load(OUTPUT_FILES["train_keys"]).astype(np.int64).tolist())
test_users_split = set(np.load(OUTPUT_FILES["test_keys"]).astype(np.int64).tolist())

for col in ["user_id", "movie_id", "rating"]:
    train_ratings[col] = pd.to_numeric(train_ratings[col], errors="coerce")
train_ratings = train_ratings.dropna(subset=["user_id", "movie_id", "rating"]).copy()
train_ratings["user_id"] = train_ratings["user_id"].astype(int)
train_ratings["movie_id"] = train_ratings["movie_id"].astype(int)

movie_content["movie_id"] = pd.to_numeric(movie_content["movie_id"], errors="coerce")
movie_content = movie_content.dropna(subset=["movie_id"]).copy()
movie_content["movie_id"] = movie_content["movie_id"].astype(int)

user_behavior["user_id"] = pd.to_numeric(user_behavior["user_id"], errors="coerce")
user_behavior = user_behavior.dropna(subset=["user_id"]).copy()
user_behavior["user_id"] = user_behavior["user_id"].astype(int)

triples = []
users_with_edge = set()

positive_train = train_ratings[train_ratings["rating"] >= POSITIVE_RATING_THRESHOLD].copy()
for row in positive_train.itertuples(index=False):
    uid = int(row.user_id)
    triples.append((f"user_{uid}", "rated", f"movie_{int(row.movie_id)}"))
    users_with_edge.add(uid)

behavior_cols_runtime = [c for c in user_behavior.columns if c not in {"user_id", "log_followers"}]
user_behavior_user_set = set(user_behavior["user_id"].astype(int).tolist())

for row in user_behavior.itertuples(index=False):
    uid = int(row.user_id)
    active_behavior_count = 0

    for behavior_col in behavior_cols_runtime:
        is_active = int(getattr(row, behavior_col))
        if is_active > 0:
            behavior_node = f"behavior_{sanitize_node_token(behavior_col)}"
            triples.append((f"user_{uid}", "has_behavior", behavior_node))
            active_behavior_count += 1
            users_with_edge.add(uid)

    # Keep users connected even when all explicit behavior flags are zero.
    if active_behavior_count == 0:
        triples.append((f"user_{uid}", "has_behavior", "behavior_no_active_flags"))
        users_with_edge.add(uid)

for row in movie_content.itertuples(index=False):
    movie_node = f"movie_{int(row.movie_id)}"

    for genre in split_tokens(getattr(row, "genres", "")):
        triples.append((movie_node, "has_genre", f"genre_{sanitize_node_token(genre)}"))

    topics = split_tokens(getattr(row, "topics", ""))[:MAX_TOPICS_PER_MOVIE]
    for topic in topics:
        triples.append((movie_node, "has_topic", f"topic_{sanitize_node_token(topic)}"))

    country_code = str(getattr(row, "country_code", "")).strip().upper()
    if country_code and country_code != "NAN":
        triples.append((movie_node, "from_country", f"country_{sanitize_node_token(country_code)}"))

# Guarantee every test user has at least one edge in KG.
fallback_edges_added = 0
fallback_missing_profile = 0
for uid in sorted(test_users_split):
    if uid in users_with_edge:
        continue

    fallback_edges_added += 1
    if uid in user_behavior_user_set:
        fallback_node = "behavior_no_active_flags"
    else:
        fallback_node = "behavior_missing_profile"
        fallback_missing_profile += 1

    triples.append((f"user_{uid}", "has_behavior", fallback_node))
    users_with_edge.add(uid)

triples_df = pd.DataFrame(triples, columns=["head", "relation", "tail"]).drop_duplicates().reset_index(drop=True)
relation_stats = triples_df.groupby("relation").size().reset_index(name="n_triples").sort_values("n_triples", ascending=False)

triples_df.to_csv(OUTPUT_FILES["kg_triples"], sep="\t", index=False)
relation_stats.to_csv(OUTPUT_FILES["kg_relation_stats"], index=False)

covered_test_users = len(test_users_split & users_with_edge)
print(f"KG triples: {len(triples_df):,}")
print(relation_stats.to_string(index=False))
print(f"Test-user KG coverage: {covered_test_users:,}/{len(test_users_split):,}")
print(f"Fallback edges added for test users: {fallback_edges_added:,} (missing-profile: {fallback_missing_profile:,})")

KG triples: 772,431
    relation  n_triples
       rated     409952
   has_topic     144124
   has_genre     138828
from_country      78626
has_behavior        901
Test-user KG coverage: 122/122
Fallback edges added for test users: 27 (missing-profile: 27)


## 7. Train KGE (TransE) and Export Entity Embeddings

In [ ]:
from pykeen.triples import TriplesFactory
from pykeen.pipeline import pipeline
import torch
import inspect

triples_df = pd.read_csv(OUTPUT_FILES["kg_triples"], sep="\t")
if triples_df.empty:
    raise ValueError("No KG triples found. Build triples before training KGE.")

labeled_triples = triples_df[["head", "relation", "tail"]].astype(str).to_numpy()
all_triples_factory = TriplesFactory.from_labeled_triples(labeled_triples)

# PyKEEN versions can return either 2-way or 3-way splits.
split_result = all_triples_factory.split(ratios=(0.8, 0.1), random_state=42)
if len(split_result) == 3:
    training_factory, testing_factory, validation_factory = split_result
elif len(split_result) == 2:
    training_factory, testing_factory = split_result
    validation_factory = None
else:
    raise ValueError(f"Unexpected split output length: {len(split_result)}")

device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
    torch.set_float32_matmul_precision("high")

    gpu_name = torch.cuda.get_device_name(0)
    total_vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    if total_vram_gb >= 10:
        kge_batch_size = 8192
    elif total_vram_gb >= 6:
        kge_batch_size = 4096
    else:
        kge_batch_size = 2048

    print(f"Using GPU for KGE: {gpu_name} ({total_vram_gb:.1f} GB), batch_size={kge_batch_size}")
else:
    kge_batch_size = 1024
    print("CUDA not available, falling back to CPU for KGE.")

pipeline_kwargs = dict(
    training=training_factory,
    testing=testing_factory,
    model="TransE",
    model_kwargs={"embedding_dim": 64, "scoring_fct_norm": 1},
    optimizer="adam",
    optimizer_kwargs={"lr": 1e-3},
    training_kwargs={"num_epochs": 100, "batch_size": kge_batch_size},
    random_seed=42,
    device=device,
)

# Keep compatibility across different PyKEEN versions.
optional_kwargs = {
    "validation": validation_factory,
    "use_tqdm": True,
    "use_tqdm_batch": False,
    "evaluation_fallback": True,
}
valid_params = inspect.signature(pipeline).parameters
for key, value in optional_kwargs.items():
    if value is not None and key in valid_params:
        pipeline_kwargs[key] = value

kge_result = pipeline(**pipeline_kwargs)

kge_model = kge_result.model
model_device = next(kge_model.parameters()).device

entity_indices = torch.arange(all_triples_factory.num_entities, device=model_device)
relation_indices = torch.arange(all_triples_factory.num_relations, device=model_device)

entity_embeddings = kge_model.entity_representations[0](indices=entity_indices).detach().cpu().numpy().astype(np.float32)
relation_embeddings = kge_model.relation_representations[0](indices=relation_indices).detach().cpu().numpy().astype(np.float32)

entity_labels = np.empty(all_triples_factory.num_entities, dtype=object)
for label, idx in all_triples_factory.entity_to_id.items():
    entity_labels[idx] = label

relation_labels = np.empty(all_triples_factory.num_relations, dtype=object)
for label, idx in all_triples_factory.relation_to_id.items():
    relation_labels[idx] = label

pd.DataFrame({
    "entity": list(all_triples_factory.entity_to_id.keys()),
    "entity_id": list(all_triples_factory.entity_to_id.values()),
}).sort_values("entity_id").to_csv(OUTPUT_FILES["kge_entity_to_id"], index=False)

pd.DataFrame({
    "relation": list(all_triples_factory.relation_to_id.keys()),
    "relation_id": list(all_triples_factory.relation_to_id.values()),
}).sort_values("relation_id").to_csv(OUTPUT_FILES["kge_relation_to_id"], index=False)

np.save(OUTPUT_FILES["kge_entity_embeddings"], entity_embeddings)
np.save(OUTPUT_FILES["kge_entity_labels"], entity_labels)

embedding_cols = [f"kge_{i:03d}" for i in range(entity_embeddings.shape[1])]

def build_prefixed_embedding_frame(prefix, id_col):
    rows = []
    for idx, label in enumerate(entity_labels.tolist()):
        if not isinstance(label, str):
            continue
        if not label.startswith(prefix):
            continue
        raw_id = label[len(prefix):]
        if not raw_id.isdigit():
            continue
        rows.append([int(raw_id)] + entity_embeddings[idx].tolist())

    if not rows:
        return pd.DataFrame(columns=[id_col] + embedding_cols)

    df = pd.DataFrame(rows, columns=[id_col] + embedding_cols)
    df = df.drop_duplicates(subset=[id_col]).sort_values(id_col).reset_index(drop=True)
    return df

item_kge_df = build_prefixed_embedding_frame("movie_", "item_id")
user_kge_df = build_prefixed_embedding_frame("user_", "user_id")

item_kge_df.to_csv(OUTPUT_FILES["kge_item_embeddings"], index=False)
user_kge_df.to_csv(OUTPUT_FILES["kge_user_embeddings"], index=False)

print(f"Entity embeddings: {entity_embeddings.shape}")
print(f"Relation embeddings: {relation_embeddings.shape}")
print(f"Item KGE vectors: {item_kge_df.shape}")
print(f"User KGE vectors: {user_kge_df.shape}")

Using GPU for KGE: NVIDIA GeForce RTX 3050 Laptop GPU (4.0 GB), batch_size=2048


Training epochs on cuda:0: 100%|██████████| 100/100 [04:26<00:00,  2.67s/epoch, loss=0.0491, prev_loss=0.0489]
Evaluating on cuda:0: 100%|██████████| 77.2k/77.2k [23:46<00:00, 54.1triple/s]
INFO:pykeen.evaluation.evaluator:Evaluation took 1568.58s seconds


Entity embeddings: (79860, 64)
Relation embeddings: (5, 64)
Item KGE vectors: (78628, 65)
User KGE vectors: (612, 65)


## 8. Ranking Evaluation Utilities

In [2]:
def sample_negatives(candidate_pool, exclude_set, n_negatives=20, rng=None):
    rng = rng or np.random.default_rng(42)
    pool = np.array([i for i in candidate_pool if i not in exclude_set], dtype=int)
    if len(pool) == 0:
        return np.array([], dtype=int)
    replace = len(pool) < n_negatives
    return rng.choice(pool, size=n_negatives, replace=replace)

def ranking_metrics_from_rank(rank, k=10):
    recall = 1.0 if rank <= k else 0.0
    ndcg = 1.0 / np.log2(rank + 1) if rank <= k else 0.0
    mrr = 1.0 / rank
    return recall, ndcg, mrr

def rank_positive_among_candidates(scores, positive_mask, rng=None):
    rng = rng or np.random.default_rng(42)
    positive_idx = int(np.flatnonzero(positive_mask)[0])
    pos_score = float(scores[positive_idx])

    better = int(np.sum(scores > pos_score))
    tie_indices = np.flatnonzero(scores == pos_score)
    tie_count = int(len(tie_indices))

    if tie_count <= 1:
        return better + 1, tie_count

    shuffled_ties = tie_indices[rng.permutation(tie_count)]
    tie_pos = int(np.where(shuffled_ties == positive_idx)[0][0])
    return better + tie_pos + 1, tie_count

def evaluate_ranker(score_fn, positives_df, candidate_pool, all_user_pos_lookup, n_negatives=20, k=10, seed=42):
    rng = np.random.default_rng(seed)
    rows = []
    tie_cases = 0

    for uid, group in positives_df.groupby("user_id"):
        uid = int(uid)
        known_positives = all_user_pos_lookup.get(uid, set())

        for _, row in group.iterrows():
            pos_item = int(row["movie_id"])
            exclude = set(known_positives)
            exclude.add(pos_item)

            negatives = sample_negatives(candidate_pool, exclude, n_negatives=n_negatives, rng=rng)
            candidate_items = np.array([pos_item] + negatives.tolist(), dtype=int)
            positive_mask = np.zeros(len(candidate_items), dtype=bool)
            positive_mask[0] = True

            perm = rng.permutation(len(candidate_items))
            candidate_items = candidate_items[perm]
            positive_mask = positive_mask[perm]

            scores = score_fn(uid, candidate_items)
            rank, tie_count = rank_positive_among_candidates(scores, positive_mask, rng=rng)
            if tie_count > 1:
                tie_cases += 1

            recall, ndcg, mrr = ranking_metrics_from_rank(rank, k=k)
            rows.append((recall, ndcg, mrr))

    if not rows:
        return {"Recall@10": np.nan, "NDCG@10": np.nan, "MRR": np.nan, "n_eval": 0, "tie_rate": np.nan}

    arr = np.array(rows, dtype=float)
    n_eval = int(len(rows))
    return {
        "Recall@10": float(arr[:, 0].mean()),
        "NDCG@10": float(arr[:, 1].mean()),
        "MRR": float(arr[:, 2].mean()),
        "n_eval": n_eval,
        "tie_rate": float(tie_cases / n_eval),
    }

## 9. Reload Exported Outputs and Build LightFM Matrices (Baseline + KGE)
This section keeps all baseline LightFM features and adds KGE vectors on top instead of replacing baseline features.

In [1]:
pip install lightfm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.4/316.4 kB 2.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for lightfm: filename=lightfm-1.17-cp311-cp311-linux_x86_64.whl size=831128 sha256=8e08b7012fd1ab73db6a8a0379ea3fe9f681f2b7a5a54cb8c92840bb9e4fb38a
  Stored in directory: /root/.cache/pip/wheels/b9/0d/8a/0729d2e6e3ca2a898ba55201f905da7db3f838a33df5b3fcdd
Successfully built lightfm


In [5]:
from lightfm.data import Dataset
from pathlib import Path
import json
import re
import numpy as np
import pandas as pd

# Helper functions needed for feature building
def split_tokens(value):
    if pd.isna(value) or value == "":
        return []
    return [t.strip() for t in str(value).split("|") if t.strip()]

def sanitize_node_token(token):
    token = str(token).strip().lower()
    token = re.sub(r"\s+", "_", token)
    token = re.sub(r"[^a-z0-9_]+", "", token)
    return token or "unknown"

# Updated: Use current directory since files are exported to /content
ARTIFACT_DIR = Path(".")

artifact_paths = {
    "ratings_clean": ARTIFACT_DIR / "01clean_ratings.csv",
    "movies_clean": ARTIFACT_DIR / "01clean_movies.csv",
    "users_clean": ARTIFACT_DIR / "01clean_users.csv",
    "movie_content_features": ARTIFACT_DIR / "02movie_content_features.csv",
    "user_behavior_features": ARTIFACT_DIR / "02user_behavior_features.csv",
    "train_ratings": ARTIFACT_DIR / "03split_train_ratings.csv",
    "test_ratings": ARTIFACT_DIR / "03split_test_ratings.csv",
    "train_keys": ARTIFACT_DIR / "03split_train_keys.npy",
    "test_keys": ARTIFACT_DIR / "03split_test_keys.npy",
    "all_item_ids": ARTIFACT_DIR / "03split_all_item_ids.npy",
    "kge_item_embeddings": ARTIFACT_DIR / "05kge_item_embeddings.csv",
    "kge_user_embeddings": ARTIFACT_DIR / "05kge_user_embeddings.csv",
}

missing_files = [path for path in artifact_paths.values() if not path.exists()]
if missing_files:
    missing_preview = "\n".join(str(p) for p in missing_files[:10])
    raise FileNotFoundError(f"Missing artifact files under {ARTIFACT_DIR.resolve()}:\n{missing_preview}")

ratings_loaded = pd.read_csv(artifact_paths["ratings_clean"])
movies_loaded = pd.read_csv(artifact_paths["movies_clean"])
users_loaded = pd.read_csv(artifact_paths["users_clean"])
movie_content_loaded = pd.read_csv(artifact_paths["movie_content_features"])
user_behavior_loaded = pd.read_csv(artifact_paths["user_behavior_features"])
train_df_loaded = pd.read_csv(artifact_paths["train_ratings"])
test_df_loaded = pd.read_csv(artifact_paths["test_ratings"])

train_users_split = set(np.load(artifact_paths["train_keys"]).astype(np.int64).tolist())
test_users_split = set(np.load(artifact_paths["test_keys"]).astype(np.int64).tolist())
all_user_ids = sorted(train_users_split | test_users_split)

all_item_ids = np.load(artifact_paths["all_item_ids"]).astype(np.int64).tolist()
all_item_set = set(int(x) for x in all_item_ids)

kge_item_df = pd.read_csv(artifact_paths["kge_item_embeddings"])
kge_user_df = pd.read_csv(artifact_paths["kge_user_embeddings"])

item_kge_cols = [c for c in kge_item_df.columns if c.startswith("kge_")]
user_kge_cols = [c for c in kge_user_df.columns if c.startswith("kge_")]

item_id_col = "item_id" if "item_id" in kge_item_df.columns else ("movie_id" if "movie_id" in kge_item_df.columns else "id")

item_kge_lookup = {
    int(row[item_id_col]): row[item_kge_cols].to_numpy(dtype=np.float32)
    for _, row in kge_item_df.iterrows()
}
user_kge_lookup = {
    int(row["user_id"]): row[user_kge_cols].to_numpy(dtype=np.float32)
    for _, row in kge_user_df.iterrows()
}

for col in ["genres", "actors", "directors"]:
    if col not in movies_loaded.columns:
        movies_loaded[col] = ""
movies_loaded["genres"] = movies_loaded["genres"].fillna("")
movies_loaded["actors"] = movies_loaded["actors"].fillna("")
movies_loaded["directors"] = movies_loaded["directors"].fillna("")
movies_loaded["year_published"] = pd.to_numeric(movies_loaded.get("year_published"), errors="coerce")

all_user_id_set = set(all_user_ids)
all_item_id_set = set(all_item_ids)

train_df_lfm = train_df_loaded[
    train_df_loaded["user_id"].astype(int).isin(all_user_id_set)
    & train_df_loaded["movie_id"].astype(int).isin(all_item_id_set)
].copy()

def build_item_feature_dicts(movies_df, item_kge_map):
    movies_local = movies_df.copy()
    year_min = movies_local["year_published"].min(skipna=True)
    year_max = movies_local["year_published"].max(skipna=True)
    year_span = (year_max - year_min) if pd.notna(year_min) and pd.notna(year_max) and year_max != year_min else 1.0
    movies_local["year_norm"] = ((movies_local["year_published"] - year_min) / year_span).fillna(0.0).clip(0.0, 1.0)

    item_feature_dicts = []
    for _, row in movies_local.iterrows():
        item_id = int(row["id"])
        feat = {}

        for g in split_tokens(row.get("genres", "")):
            feat[f"genre__{g}"] = 1.0
        for a in split_tokens(row.get("actors", "")):
            feat[f"actor__{a}"] = 1.0
        for d in split_tokens(row.get("directors", "")):
            feat[f"director__{d}"] = 1.0
        feat["year_norm"] = float(row["year_norm"])

        kge_vector = item_kge_map.get(item_id)
        if kge_vector is not None:
            for idx, value in enumerate(kge_vector):
                feat[f"kge_item__{idx:03d}"] = float(value)

        item_feature_dicts.append((item_id, feat))

    return item_feature_dicts

def build_user_feature_dicts(all_user_ids_list, user_behavior_df, user_kge_map):
    baseline_flags = [c for c in ["early_bird", "night_owl", "weekend_tweeter"] if c in user_behavior_df.columns]

    behavior_lookup = {}
    for _, row in user_behavior_df.iterrows():
        uid = int(row["user_id"])
        behavior_lookup[uid] = row

    user_feature_dicts = []
    for uid in all_user_ids_list:
        row = behavior_lookup.get(uid)
        feat = {}

        for flag in baseline_flags:
            raw_value = 0 if row is None else row.get(flag, 0)
            feat[f"flag__{flag}"] = float(int(0 if pd.isna(raw_value) else raw_value))

        raw_followers = 0.0 if row is None else row.get("log_followers", 0.0)
        feat["log_followers"] = float(0.0 if pd.isna(raw_followers) else raw_followers)

        kge_vector = user_kge_map.get(uid)
        if kge_vector is not None:
            for idx, value in enumerate(kge_vector):
                feat[f"kge_user__{idx:03d}"] = float(value)

        user_feature_dicts.append((uid, feat))

    return user_feature_dicts

def prepare_interactions(df):
    return [(int(r["user_id"]), int(r["movie_id"])) for _, r in df.iterrows()]

item_features = build_item_feature_dicts(movies_loaded, item_kge_lookup)
user_features = build_user_feature_dicts(all_user_ids, user_behavior_loaded, user_kge_lookup)

dataset = Dataset()
dataset.fit(
    users=all_user_ids,
    items=all_item_ids,
    user_features=sorted({f for _, d in user_features for f in d.keys()}),
    item_features=sorted({f for _, d in item_features for f in d.keys()}),
)

train_interactions, _ = dataset.build_interactions(prepare_interactions(train_df_lfm))
user_features_mx = dataset.build_user_features(user_features)
item_features_mx = dataset.build_item_features(item_features)
user_id_map, _, item_id_map, _ = dataset.mapping()

kge_user_train_cov = len(train_users_split & set(user_kge_lookup.keys())) / max(1, len(train_users_split))
kge_user_test_cov = len(test_users_split & set(user_kge_lookup.keys())) / max(1, len(test_users_split))

print(f"Artifact dir: {ARTIFACT_DIR.resolve()}")
print(f"LightFM dataset built successfully. Users: {len(user_id_map)}, Items: {len(item_id_map)}")
print(f"KGE user coverage -> train: {kge_user_train_cov:.3f}, test: {kge_user_test_cov:.3f}")

Artifact dir: /content
LightFM dataset built successfully. Users: 611, Items: 78628
KGE user coverage -> train: 1.000, test: 1.000


## 10. Train LightFM (Baseline Configuration)

In [6]:
from lightfm import LightFM

model = LightFM(loss="warp", no_components=64, learning_rate=0.05, random_state=42)
model.fit(
    train_interactions,
    user_features=user_features_mx,
    item_features=item_features_mx,
    epochs=30,
    num_threads=1,
    verbose=True,
)

Epoch: 100%|██████████| 30/30 [59:54<00:00, 119.82s/it]


## 11. Evaluate on User-Split Test Set

In [7]:
# Build the missing 'all_user_pos' lookup dictionary
def build_user_pos_lookup(train_df, test_df):
    lookup = {}
    combined_df = pd.concat([train_df, test_df])
    for uid, group in combined_df.groupby('user_id'):
        lookup[int(uid)] = set(group['movie_id'].astype(int).tolist())
    return lookup

all_user_pos = build_user_pos_lookup(train_df_loaded, test_df_loaded)

def lfm_score_fn(uid, candidate_items):
    if uid not in user_id_map:
        return np.zeros(len(candidate_items), dtype=float)

    uidx = user_id_map[uid]
    item_idxs = []
    valid_positions = []

    for pos, item in enumerate(candidate_items):
        if item in item_id_map:
            item_idxs.append(item_id_map[item])
            valid_positions.append(pos)

    scores = np.zeros(len(candidate_items), dtype=float)
    if item_idxs:
        preds = model.predict(
            uidx,
            np.array(item_idxs, dtype=int),
            user_features=user_features_mx,
            item_features=item_features_mx,
        )
        for pos, sc in zip(valid_positions, preds):
            scores[pos] = float(sc)

    return scores

test_metrics = evaluate_ranker(
    lfm_score_fn,
    positives_df=test_df_loaded,
    candidate_pool=all_item_set,
    all_user_pos_lookup=all_user_pos,
    n_negatives=20,
    k=10,
    seed=43,
)

# Ensure OUTPUT_FILES or equivalent path exists for saving
metrics_path = Path("06lightfm_test_metrics.json")
with open(metrics_path, "w", encoding="utf-8") as f:
    json.dump(test_metrics, f, indent=2)

print("LightFM + KGE (User Split 80/20)")
print("Test:", test_metrics)
print(f"Saved metrics to: {metrics_path}")

LightFM + KGE (User Split 80/20)
Test: {'Recall@10': 0.8846368259903771, 'NDCG@10': 0.6817043170202036, 'MRR': 0.6242506441045962, 'n_eval': 233194, 'tie_rate': 2.5729649990994622e-05}
Saved metrics to: 06lightfm_test_metrics.json
